# 🚀 LLM in Python - From Simple to Complex

This notebook walks through practical Groq API examples, progressively building in complexity.

| # | Example | Difficulty |
|---|---------|------------|
| 1 | Setup & Basic Chat | ⭐ |
| 2 | System Prompts & Personas | ⭐⭐ |
| 3 | Streaming Responses | ⭐⭐ |
| 6 | Sentiment Analysis Pipeline | ⭐⭐⭐ |

**> Get your free API key at:** [https://console.groq.com/keys]

## 🔧 Setup — Install & Configure

In [1]:
!pip install groq faiss-cpu sentence-transformers ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.0 MB/s eta 0:00:00


In [4]:
import os
from google.colab import userdata

# Option A: Store your key in Colab Secrets (recommended)
# Go to the key icon in the left sidebar and add GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('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')
    print('Loaded API key from Colab Secrets')
except Exception:
    GROQ_API_KEY = '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'  # Replace with your key
    print('Using hardcoded key - prefer Colab Secrets for security')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

Using hardcoded key - prefer Colab Secrets for security


In [5]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    'fast':     'llama-3.1-8b-instant',
    'balanced': 'llama-3.3-70b-versatile',
    'code':     'qwen-2.5-coder-32b',
}

print('Groq client ready!')
print('Models:', list(MODELS.keys()))

Groq client ready!
Models: ['fast', 'balanced', 'code']


---
## Example 1 - Basic Chat Completion

In [6]:
response = client.chat.completions.create(
    model=MODELS['fast'],
    messages=[
        {"role": "user", "content": "What is Groq and why is it so fast?"}
    ]
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")
print(f"Speed: {response.usage.completion_tokens / response.usage.completion_time:.0f} tokens/sec")

Groq is a US-based company that specializes in the development of high-performance and low-power compute chips, particularly focused on the field of Tensor Processing Units (TPUs). 

TPUs were first developed by Google, a key collaborator and investor in Groq. The core idea behind TPUs is to efficiently process machine learning (ML) computations, like matrix multiplications, which are crucial to training deep learning models.

TPUs offer a number of benefits over traditional CPUs and graphical processing units (GPUs). Some of these benefits include:

1. **Massive parallel execution capabilities**: TPU cores can execute parallel threads of code far more efficiently than their CPU and GPU counterparts.
2. **Tensor-based architecture**: TPUs are designed specifically for matrix operations (tensors) and have hardware units to accelerate these operations.

Groq's chips leverage the same general architecture and principles of TPUs, and are intended to offer high performance and low power con

---
## Example 2 - System Prompts & Personas

In [7]:
def chat_with_persona(persona_prompt, user_message, model=None):
    model = model or MODELS['fast']
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": persona_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


socratic = """
You are a Socratic teacher. Never give direct answers.
Guide the student with thoughtful questions only. Keep responses under 80 words.
"""
print("Socratic Teacher:")
print(chat_with_persona(socratic, "Why does the sky appear blue?"))

print("\n" + "-"*60 + "\n")

pirate_chef = """
You are a salty old pirate who is also a Michelin-star chef.
Speak in pirate dialect but give genuine, expert cooking advice.
"""
print("Pirate Chef:")
print(chat_with_persona(pirate_chef, "How do I make the perfect risotto?"))

Socratic Teacher:
What do you think happens when sunlight enters Earth's atmosphere? Does it simply pass through untouched, or does it interact with something in its path? What properties of light might be affected by this interaction? Can you think of any objects or materials that appear blue due to their interaction with light?

------------------------------------------------------------

Pirate Chef:
Arrrr, ye landlubber be wantin' to know the secrets o' the perfect risotto, eh? Alright then, listen close and I'll share me treasure with ye. But first, let me warn ye: makin' a proper risotto be a swashbucklin' task that requires patience, skill, and a bit o' flair.

First, ye'll need to gather yer ingredients like a scurvy dog gatherin' treasure:

* 1 cup o' Arborio rice (the good stuff, not that other rubbish)
* 4 cups o' vegetable or chicken broth, warmed and ready to go
* 2 tablespoons o' olive oil
* 1 small onion, finely chopped
* 2 cloves o' garlic, minced
* 1 cup o' white wine

---
## Example 3 - Sentiment Analysis Pipeline

Process a batch of customer reviews, classify each with fine-grained sentiment, extract topics, and produce a summary report.

In [9]:
import json
import pandas as pd

reviews = [
    "Absolutely love this product! Fast shipping and great quality. Will buy again.",
    "Terrible experience. Item arrived broken and customer service was unhelpful.",
    "It's okay. Does what it's supposed to do but nothing special.",
    "Amazing quality for the price! Blew my expectations out of the water.",
    "Returned it after 2 days. Poor build quality and the size was wrong.",
    "Decent product but delivery took 3 weeks longer than expected.",
    "Five stars! The packaging was beautiful and the item works perfectly.",
    "Not bad, not great. Instructions were confusing but it works fine now.",
]

def analyze_sentiment(review):
    response = client.chat.completions.create(
        model=MODELS["fast"],
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a sentiment analysis engine.\n\n"
                    "Return ONLY a JSON object with these exact keys:\n"

                    '"sentiment": one of ["very_positive", "positive", "neutral", "negative", "very_negative"]\n'

                    "Use these sentiment rules strictly:\n"
                    "- positive: generally satisfied, good overall, minor praise without extreme enthusiasm\n"
                    "- neutral: mixed, balanced, or plainly factual with little emotion\n"
                    "- negative: generally dissatisfied, some criticism, but not strongly emotional or catastrophic\n"

                )
            },
            {"role": "user", "content": review}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    result = json.loads(response.choices[0].message.content)
    result["review"] = review
    return result

results = [analyze_sentiment(r) for r in reviews]


df = pd.DataFrame(results)

df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()


df = df[["review", "sentiment"]]

df

,review,sentiment
0,Absolutely love this product! Fast shipping an...,very_positive
1,Terrible experience. Item arrived broken and c...,very_negative
2,It's okay. Does what it's supposed to do but n...,neutral
3,Amazing quality for the price! Blew my expecta...,very_positive
4,Returned it after 2 days. Poor build quality a...,negative
5,Decent product but delivery took 3 weeks longe...,negative
6,Five stars! The packaging was beautiful and th...,very_positive
7,"Not bad, not great. Instructions were confusin...",neutral


---
## Quick Reference


**Useful links:**
- [Groq Docs](https://console.groq.com/docs)
- [API Keys](https://console.groq.com/keys)
- [Model List](https://console.groq.com/docs/models)
- [Discord Community](https://discord.gg/groq)